# Dose Whisperer — Notebook 1: Data preparation and label evidence

**Research prototype. Not for clinical use.** Flags are prototype safety signals, not medical advice.

This notebook covers Stages 0–3 of the project:

| Stage | What it does |
|---|---|
| 0 | Checks the environment (GPU, files) and profiles `PRESCRIPTIONS.csv` |
| 1 | Builds an offline openFDA label cache for the 12 curated drugs |
| 2 | Cross-checks MIMIC prescriptions against the label cache |
| 3 | Mines the standard dosing sentences per drug and route (consensus across labels) |

Run the cells **from top to bottom**. Stage 1 is slow (it reads ~8 GB of JSON); it is skipped automatically if the cache already exists.

## 0. Configuration
Edit only the paths in this cell if your folders differ.

In [1]:
import os, io, re, gc, json, csv
from pathlib import Path
from collections import defaultdict
from contextlib import redirect_stdout

import pandas as pd

# ---- paths (override with environment variables if needed) ----
DATA_DIR    = Path(os.environ.get("DW_DATA_DIR", Path.home() / "dataset"))
OPENFDA_DIR = DATA_DIR / "openfda"
PROJECT_DIR = Path(os.environ.get("DW_PROJECT_DIR", Path.home() / "SOURABH_S"))
CACHE_DIR   = PROJECT_DIR / "data" / "label_cache"
PROFILE_DIR = PROJECT_DIR / "outputs" / "profile"
for d in (CACHE_DIR, PROFILE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- the 12 curated drugs: key -> substring required in the label's single generic name ----
CURATED = {
    "furosemide": "furosemide", "heparin": "heparin", "metoprolol": "metoprolol",
    "morphine": "morphine", "acetaminophen": "acetaminophen", "vancomycin": "vancomycin",
    "lorazepam": "lorazepam", "fentanyl": "fentanyl", "prednisone": "prednisone",
    "warfarin": "warfarin", "potassium_chloride": "potassium chloride",
    "magnesium_sulfate": "magnesium sulfate",
}

def save_and_print(fn, out_path):
    """Run fn(), show its printed output, and also save it to a text file."""
    buf = io.StringIO()
    with redirect_stdout(buf):
        fn()
    text = buf.getvalue()
    Path(out_path).write_text(text, encoding="utf-8")
    print(text)
    print(f"[saved to {out_path}]")

print("DATA_DIR   :", DATA_DIR)
print("PROJECT_DIR:", PROJECT_DIR)
print("data dir exists:", DATA_DIR.exists())

DATA_DIR   : /home/user9/dataset
PROJECT_DIR: /home/user9/SOURABH_S
data dir exists: True


## Stage 0 — Environment check and prescription profile
Confirms the five MIMIC files and the openFDA shards are present, checks the GPU, and profiles the prescriptions table.

In [7]:
NEEDED = ["PRESCRIPTIONS.csv", "DIAGNOSES_ICD.csv", "D_ICD_DIAGNOSES.csv", "ADMISSIONS.csv", "PATIENTS.csv"]

def human(n):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} PB"

print("=== Required MIMIC files ===")
for n in NEEDED:
    p = DATA_DIR / n
    print(f"  {n:<22}", f"FOUND ({human(p.stat().st_size)})" if p.exists() else "MISSING")

shards = sorted(p for p in OPENFDA_DIR.rglob("drug-label-*.json") if p.is_file()) if OPENFDA_DIR.exists() else []
print(f"\nopenFDA shards found: {len(shards)} (expected 14), total {human(sum(s.stat().st_size for s in shards))}")

print("\n=== GPU ===")
try:
    import torch
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print(f"{props.name}, {props.total_memory/1024**3:.0f} GB VRAM, torch {torch.__version__}, CUDA {torch.version.cuda}")
    else:
        print("CUDA not available to PyTorch.")
except ImportError:
    print("torch not installed in this kernel.")

=== Required MIMIC files ===
  PRESCRIPTIONS.csv      FOUND (1.6 MB)
  DIAGNOSES_ICD.csv      FOUND (49.6 KB)
  D_ICD_DIAGNOSES.csv    FOUND (1.3 MB)
  ADMISSIONS.csv         FOUND (26.3 KB)
  PATIENTS.csv           FOUND (8.5 KB)

openFDA shards found: 14 (expected 14), total 8.2 GB

=== GPU ===
NVIDIA A100 80GB PCIe, 79 GB VRAM, torch 2.14.1+cu130, CUDA 13.0


In [8]:
rx = pd.read_csv(DATA_DIR / "PRESCRIPTIONS.csv", low_memory=False)
rx.columns = [c.upper() for c in rx.columns]
print(f"Rows: {len(rx):,} | Patients: {rx['SUBJECT_ID'].nunique()} | Admissions: {rx['HADM_ID'].nunique()}")
print("Columns:", list(rx.columns))
print("NOTE: MIMIC-III PRESCRIPTIONS has no frequency column; frequency must be assigned later.")

drug_col = "DRUG_NAME_GENERIC" if "DRUG_NAME_GENERIC" in rx.columns else "DRUG"
rx["_drug"] = rx[drug_col].fillna(rx["DRUG"]).astype(str).str.strip().str.lower()
top30 = rx["_drug"].value_counts().head(30)
top30.rename_axis("drug").reset_index(name="rows").to_csv(PROFILE_DIR / "top30_drugs.csv", index=False)
display(top30.to_frame("rows"))

print("\nTop dose units:")
display(rx["DOSE_UNIT_RX"].value_counts().head(10).to_frame("rows"))
nonnum = pd.to_numeric(rx["DOSE_VAL_RX"], errors="coerce").isna() & rx["DOSE_VAL_RX"].notna()
print(f"Non-numeric DOSE_VAL_RX (ranges/text): {nonnum.sum():,} rows ({nonnum.mean():.1%})")

Rows: 10,398 | Patients: 94 | Admissions: 122
Columns: ['ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'STARTDATE', 'ENDDATE', 'DRUG_TYPE', 'DRUG', 'DRUG_NAME_POE', 'DRUG_NAME_GENERIC', 'FORMULARY_DRUG_CD', 'GSN', 'NDC', 'PROD_STRENGTH', 'DOSE_VAL_RX', 'DOSE_UNIT_RX', 'FORM_VAL_DISP', 'FORM_UNIT_DISP', 'ROUTE']
NOTE: MIMIC-III PRESCRIPTIONS has no frequency column; frequency must be assigned later.


,rows
_drug,
potassium chloride,461
d5w,439
0.9% sodium chloride,409
ns,362
furosemide,346
iso-osmotic dextrose,265
5% dextrose,256
sw,244
heparin sodium,223



Top dose units:


,rows
DOSE_UNIT_RX,
mg,4378
mL,1849
ml,1216
mEq,670
UNIT,588
gm,488
TAB,174
NEB,153
g,148


Non-numeric DOSE_VAL_RX (ranges/text): 655 rows (6.3%)


## Stage 1 — Build the offline openFDA label cache
Reads each shard **one at a time**, keeps single-ingredient labels for the 12 curated drugs, and stores the sections we need as evidence.
Each shard needs several GB of RAM while loaded. Set `FORCE_REBUILD = True` to rebuild even if the cache exists.

In [9]:
FORCE_REBUILD = True

SECTIONS = ["indications_and_usage", "dosage_and_administration", "dosage_forms_and_strengths",
            "contraindications", "boxed_warning", "warnings", "warnings_and_cautions",
            "precautions", "use_in_specific_populations", "overdosage"]
MAX_CHARS = 30000

def clean(val):
    if isinstance(val, list):
        val = "\n".join(str(v) for v in val)
    return str(val).strip()[:MAX_CHARS]

BAD_NAME_TERMS = [" and ", " in ", " with ", ",", "/", "+"]

def match_drug(label):
    ofda = label.get("openfda", {})
    names = [n.lower() for n in ofda.get("generic_name", [])]
    subs  = [s.lower() for s in ofda.get("substance_name", [])]
    if len(names) != 1 or len(subs) > 1:      # combination products have several active substances
        return None
    gname = names[0]
    if any(t in gname for t in BAD_NAME_TERMS):
        return None
    check = subs[0] if subs else gname
    for key, sub in CURATED.items():
        if sub in check:
            return key
    return None

def build_cache():
    shards = sorted(p for p in OPENFDA_DIR.rglob("drug-label-*.json") if p.is_file())
    assert shards, f"No shards found in {OPENFDA_DIR}"
    kept, total = defaultdict(list), 0
    for i, shard in enumerate(shards, 1):
        print(f"[{i}/{len(shards)}] reading {shard.name} ...", flush=True)
        with open(shard, "r", encoding="utf-8") as f:
            data = json.load(f)
        results = data.get("results", [])
        total += len(results)
        for label in results:
            key = match_drug(label)
            if key is None:
                continue
            ofda = label.get("openfda", {})
            kept[key].append({
                "id": label.get("id"), "set_id": label.get("set_id"),
                "effective_time": label.get("effective_time"),
                "brand_name": ofda.get("brand_name", []),
                "generic_name": ofda.get("generic_name", []),
                "substance_name": ofda.get("substance_name", []),
                "manufacturer_name": ofda.get("manufacturer_name", []),
                "route": ofda.get("route", []),
                "product_type": ofda.get("product_type", []),
                "source_shard": shard.name,
                "sections": {s: clean(label[s]) for s in SECTIONS if s in label},
            })
        del data, results
        gc.collect()

    rows = []
    for key in CURATED:
        recs = sorted(kept.get(key, []), key=lambda r: r.get("effective_time") or "", reverse=True)
        with open(CACHE_DIR / f"{key}.json", "w", encoding="utf-8") as f:
            json.dump(recs, f, ensure_ascii=False, indent=1)
        row = {"drug": key, "labels": len(recs)}
        row.update({s: sum(1 for r in recs if s in r["sections"]) for s in SECTIONS})
        rows.append(row)
    cov = pd.DataFrame(rows)
    cov.to_csv(PROFILE_DIR / "label_coverage.csv", index=False)
    print(f"\nTotal labels scanned: {total:,}")
    return cov

have_cache = all((CACHE_DIR / f"{k}.json").exists() for k in CURATED)
if have_cache and not FORCE_REBUILD:
    print("Cache already exists; skipping rebuild (set FORCE_REBUILD = True to redo).")
else:
    coverage = build_cache()

[1/14] reading drug-label-0001-of-0014.json ...
[2/14] reading drug-label-0002-of-0014.json ...
[3/14] reading drug-label-0003-of-0014.json ...
[4/14] reading drug-label-0004-of-0014.json ...
[5/14] reading drug-label-0005-of-0014.json ...
[6/14] reading drug-label-0006-of-0014.json ...
[7/14] reading drug-label-0007-of-0014.json ...
[8/14] reading drug-label-0008-of-0014.json ...
[9/14] reading drug-label-0009-of-0014.json ...
[10/14] reading drug-label-0010-of-0014.json ...
[11/14] reading drug-label-0011-of-0014.json ...
[12/14] reading drug-label-0012-of-0014.json ...
[13/14] reading drug-label-0013-of-0014.json ...
[14/14] reading drug-label-0014-of-0014.json ...

Total labels scanned: 262,888


In [10]:
coverage = pd.read_csv(PROFILE_DIR / "label_coverage.csv")
display(coverage[["drug", "labels", "dosage_and_administration", "contraindications",
                  "boxed_warning", "warnings", "warnings_and_cautions", "precautions"]])

,drug,labels,dosage_and_administration,contraindications,boxed_warning,warnings,warnings_and_cautions,precautions
0,furosemide,147,145,145,23,117,30,132
1,heparin,56,56,56,0,5,51,5
2,metoprolol,346,346,346,15,19,327,18
3,morphine,76,76,76,71,4,72,4
4,acetaminophen,1015,1015,21,21,994,32,0
5,vancomycin,80,80,80,2,48,32,48
6,lorazepam,102,99,102,94,100,2,100
7,fentanyl,13,13,13,13,0,13,0
8,prednisone,240,240,239,0,233,7,233
9,warfarin,75,75,75,75,0,75,0


## Stage 2 — Cross-check MIMIC prescriptions against the label cache
For each drug: how it appears in MIMIC (names, routes, units, dose spread) and how complete its label evidence is.
Note the exclusions: combination products (e.g. oxycodone-acetaminophen) and heparin flushes are **not** therapeutic doses of the curated drug.

In [11]:
EXCLUDE_NAME_TERMS = {
    "acetaminophen": ["oxycodone", "hydrocodone", "codeine", "butalbital", "tramadol"],
    "heparin": ["flush", "lock"],
}
SECS = ["dosage_and_administration", "contraindications", "boxed_warning", "warnings",
        "warnings_and_cautions", "precautions"]

rx["_name"] = rx["DRUG_NAME_GENERIC"].fillna(rx["DRUG"]).astype(str).str.lower()
rx["_dose"] = pd.to_numeric(rx["DOSE_VAL_RX"], errors="coerce")

def mimic_rows(key):
    sub = CURATED[key]
    df = rx[rx["_name"].str.contains(sub, regex=False)]
    for t in EXCLUDE_NAME_TERMS.get(key, []):
        df = df[~df["_name"].str.contains(t, regex=False)]
    return df

def inspect_all():
    for key in CURATED:
        print("=" * 78)
        print(key.upper())
        labels = json.load(open(CACHE_DIR / f"{key}.json", encoding="utf-8"))
        print(f"labels: {len(labels)}")
        print("coverage:", ", ".join(f"{s}={sum(1 for l in labels if s in l['sections'])}" for s in SECS))
        sub_rx = mimic_rows(key)
        print(f"MIMIC rows (after exclusions): {len(sub_rx)} | patients: {sub_rx['SUBJECT_ID'].nunique()}")
        print("names :", sub_rx["_name"].value_counts().head(4).to_dict())
        print("routes:", sub_rx["ROUTE"].value_counts().head(6).to_dict())
        print("units :", sub_rx["DOSE_UNIT_RX"].value_counts().head(5).to_dict())
        for unit, g in sub_rx.groupby("DOSE_UNIT_RX"):
            d = g["_dose"].dropna()
            if len(g) >= 10 and len(d):
                print(f"  dose [{unit}]: n={len(d)} min={d.min():g} median={d.median():g} max={d.max():g}")

save_and_print(inspect_all, PROFILE_DIR / "inspection.txt")

FUROSEMIDE
labels: 147
coverage: dosage_and_administration=145, contraindications=145, boxed_warning=23, warnings=117, warnings_and_cautions=30, precautions=132
MIMIC rows (after exclusions): 346 | patients: 52
names : {'furosemide': 346}
routes: {'IV': 257, 'PO': 46, 'IV DRIP': 39, 'PO/NG': 3, 'NG': 1}
units : {'mg': 346}
  dose [mg]: n=346 min=5 median=20 max=200
HEPARIN
labels: 56
coverage: dosage_and_administration=56, contraindications=56, boxed_warning=0, warnings=5, warnings_and_cautions=51, precautions=5
MIMIC rows (after exclusions): 225 | patients: 75
names : {'heparin sodium': 223, 'heparin': 2}
routes: {'IV': 123, 'SC': 99, 'DWELL': 3}
units : {'UNIT': 225}
  dose [UNIT]: n=134 min=300 median=5000 max=25000
METOPROLOL
labels: 346
coverage: dosage_and_administration=346, contraindications=346, boxed_warning=15, warnings=19, warnings_and_cautions=327, precautions=18
MIMIC rows (after exclusions): 247 | patients: 51
names : {'metoprolol tartrate': 138, 'metoprolol': 101, 'meto

## Stage 3 — Standard dosing sentences by consensus
For each drug and route group, counts in how many **different labels** each dosing sentence appears, after drug-specific filters
(e.g. metoprolol tartrate only, no extended-release or catheter-lock products). Pediatric sentences are dropped because the
checker will cover adults only. A sentence found in many labels is a reliable one to quote in the curated dose table.

In [14]:
FILTERS = {
    "furosemide": ([], []),
    "heparin": ([], ["defencath"]),
    "metoprolol": (["tartrate"], ["succinate", "extended-release", "extended release"]),
    "morphine": ([], []),
    "acetaminophen": ([], []), "vancomycin": ([], []), "lorazepam": ([], []),
    "fentanyl": ([], []), "prednisone": ([], []), "warfarin": ([], []),
    "potassium_chloride": ([], []), "magnesium_sulfate": ([], []),
}
ROUTE_GROUPS = ["ORAL", "INTRAVENOUS", "SUBCUTANEOUS", "INTRAMUSCULAR", "RECTAL", "TRANSDERMAL"]
TOP_N = 14

DOSE_RE  = re.compile(r"\d[\d.,]*\s*(?:to|-|–)?\s*\d*[\d.,]*\s*(?:mg|mcg|µg|g|gm|units?|mEq|mL|ml)\b", re.I)
LIMIT_RE = re.compile(r"(per day|daily|24 hours|24-hour|maximum|not exceed|exceed|usual|initial|every \d)", re.I)
PED_RE   = re.compile(r"(pediatric|paediatric|child|infant|neonat|newborn|/kg|per kg|kilogram)", re.I)

def norm_sentence(s):
    s = re.sub(r"\(\s*[\d.,\s]+\)", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def dose_sentences(text):
    text = re.sub(r"\s+", " ", text)
    for s in re.split(r"(?<=[.;])\s+", text):
        s = norm_sentence(s)
        if 20 < len(s) < 400 and DOSE_RE.search(s) and LIMIT_RE.search(s) and not PED_RE.search(s):
            yield s

def mine_all():
    for drug, (inc, exc) in FILTERS.items():
        labels = json.load(open(CACHE_DIR / f"{drug}.json", encoding="utf-8"))
        kept = []
        for l in labels:
            d = l["sections"].get("dosage_and_administration", "").lower()
            if not d or (inc and not any(t in d for t in inc)) or any(t in d for t in exc):
                continue
            kept.append(l)
        print("=" * 78)
        print(f"{drug.upper()} | labels after filter: {len(kept)} of {len(labels)} | include={inc} exclude={exc}")
        by_route = defaultdict(list)
        for l in kept:
            for r in (l["route"] or ["?"]):
                by_route[r].append(l)
        for route in ROUTE_GROUPS:
            group = by_route.get(route, [])
            counts = defaultdict(set)
            for l in group:
                for s in set(dose_sentences(l["sections"]["dosage_and_administration"])):
                    counts[s].add(l["set_id"])
            if not counts:
                continue
            print(f"\n  [{route}] {len(group)} labels")
            for s, ids in sorted(counts.items(), key=lambda kv: -len(kv[1]))[:TOP_N]:
                print(f"   {len(ids):>4}x  {s}")

save_and_print(mine_all, PROFILE_DIR / "dose_consensus.txt")

FUROSEMIDE | labels after filter: 145 of 147 | include=[] exclude=[]

  [ORAL] 97 labels
     40x  When doses exceeding 80 mg/day are given for prolonged periods, careful clinical observation and laboratory monitoring are particularly advisable (see PRECAUTIONS: Laboratory Tests ).
     27x  When doses exceeding 80 mg/day are given for prolonged periods, careful clinical observation and laboratory monitoring are particularly advisable.
     22x  Adults The usual initial dose of Furosemide tablets for hypertension is 80 mg, usually divided into 40 mg twice a day.
     22x  Adults The usual initial dose of Furosemide tablets is 20 to 80 mg given as a single dose.
     17x  Adults - The usual initial dose of furosemide tablets for hypertension is 80 mg, usually divided into 40 mg twice a day.
     17x  Adults - The usual initial dose of furosemide tablets is 20 to 80 mg given as a single dose.
     16x  Adults --The usual initial dose of furosemide for hypertension is 80mg, usually divide

In [15]:
# --- Diagnostic: adult 24-hour limit statements in oral acetaminophen labels ---
acet = json.load(open(CACHE_DIR / "acetaminophen.json", encoding="utf-8"))
oral = [l for l in acet if "ORAL" in l["route"] and "dosage_and_administration" in l["sections"]]
print("oral labels:", len(oral),
      "| Rx:", sum("PRESCRIPTION" in " ".join(l["product_type"]) for l in oral),
      "| OTC:", sum("OTC" in " ".join(l["product_type"]) for l in oral))

pat = re.compile(r"[^.;]*\b(?:24 hours?|24-hour|per day|daily)\b[^.;]*", re.I)
cnt = defaultdict(set)
for l in oral:
    text = re.sub(r"\s+", " ", l["sections"]["dosage_and_administration"])
    for m in pat.finditer(text):
        s = m.group(0).strip()
        low = s.lower()
        if (re.search(r"(adult|12 years|tablet|caplet|capsule|\d,?\d{3}\s*mg)", low)
                and not re.search(r"(under 12|children under|weight|age \(yr|dose \(ml\)|\bml\b)", low)
                and 15 < len(s) < 250):
            cnt[re.sub(r"\s+", " ", low)].add(l["set_id"])
for s, ids in sorted(cnt.items(), key=lambda kv: -len(kv[1]))[:25]:
    print(f"{len(ids):>4}x  {s}")

oral labels: 978 | Rx: 0 | OTC: 978
  23x  do not crush, chew, split or dissolve • do not take more than 6 caplets in 24 hours • do not use for more than 10 days unless directed by a doctor under 18 years of age • ask a doctor
  17x  do not crush, chew, split or dissolve do not take more than 6 caplets in 24 hours, unless directed by a doctor do not use for more than 10 days unless directed by a doctor under 18 years of age ask a doctor
  12x  do not crush, chew, split or dissolve do not take more than 6 tablets in 24 hours do not use for more than 10 days unless directed by a doctor
  10x  do not crush, chew, split or dissolve ■ do not take more than 6 caplets in 24 hours ■ do not use for more than 10 days unless directed by a doctor under 18 years of age ■ ask a doctor
  10x  do not take more than 8 tablets in 24 hours
   9x  do not crush, chew, split or dissolve do not take more than 6 caplets in 24 hours do not use for more than 10 days unless directed by a doctor under 18 years of

In [16]:
# ===== Stage 4: curated dose-limit table + evidence verification =====
import re, json
import pandas as pd

def norm(s):
    """Normalise text so label wording variants match (1,000 -> 1000, '20 mg to 40 mg' -> '20 to 40 mg', etc.)."""
    s = s.lower()
    s = re.sub(r"(\d),(\d{3})", r"\1\2", s)
    s = re.sub(r"(\d)\s*[-–]\s*(\d)", r"\1 to \2", s)
    s = re.sub(r"(\d)\s*(?:mcg|mg|gm|grams?|g|meq|units?)(?:/\w+)?\s+to\s+(?=\d)", r"\1 to ", s)
    s = re.sub(r"(?<!\d)\.|\.(?!\d)", " ", s)
    s = re.sub(r"[^a-z0-9./%]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

# same label filters as Stage 3 (include any / exclude any), applied to the dosage section
LABEL_FILTERS = {
    "metoprolol": (["tartrate"], ["succinate", "extended-release", "extended release"]),
    "heparin": ([], ["defencath"]),
}

# MIMIC route code -> label route group ("INFUSION" = rate/titration, NOT checked by the dose rule)
ROUTE_MAP = {"IV": "INTRAVENOUS", "IV DRIP": "INFUSION", "PO": "ORAL", "PO/NG": "ORAL", "NG": "ORAL",
             "SC": "SUBCUTANEOUS", "IM": "INTRAMUSCULAR", "PR": "RECTAL", "TP": "TRANSDERMAL", "SL": "SUBLINGUAL"}

# unit conversions inside the mass family
MASS_TO_MG = {"mcg": 0.001, "mg": 1.0, "g": 1000.0, "gm": 1000.0}

# scope: "dose" = one administration, "day" = total per 24 h (dose x assigned frequency)
# limits: field -> (value, supporting label quote). usual_* => WARNING outside; hard_high => HIGH above.
DOSE_TABLE = [
 dict(drug="furosemide", route="ORAL", scope="dose", unit="mg", limits={
    "usual_low":  (20, "usual initial dose of furosemide tablets is 20 to 80 mg given as a single dose"),
    "usual_high": (80, "usual initial dose of furosemide tablets is 20 to 80 mg given as a single dose")}),
 dict(drug="furosemide", route="ORAL", scope="day", unit="mg", limits={
    "usual_high": (80, "doses exceeding 80 mg/day are given for prolonged periods"),
    "hard_high":  (600, "titrated up to 600 mg/day")}),
 dict(drug="furosemide", route="INTRAVENOUS", scope="dose", unit="mg", limits={
    "usual_low":  (20, "usual initial dose of furosemide is 20 to 40 mg given as a single dose"),
    "usual_high": (40, "usual initial dose of furosemide is 20 to 40 mg given as a single dose")}),
 dict(drug="metoprolol", route="ORAL", scope="day", unit="mg", limits={
    "usual_low":  (100, "dosage range of metoprolol tartrate tablets is 100 to 400 mg per day"),
    "usual_high": (400, "dosage range of metoprolol tartrate tablets is 100 to 400 mg per day"),
    "hard_high":  (450, "dosage range of metoprolol tartrate tablets is 100 to 450 mg per day")}),
 dict(drug="metoprolol", route="INTRAVENOUS", scope="dose", unit="mg", limits={
    "usual_high": (5, "three bolus injections of 5 mg")}),
 dict(drug="vancomycin", route="INTRAVENOUS", scope="dose", unit="mg", limits={
    "usual_low":  (500, "500 mg every 6 hours or 1 g every 12 hours"),
    "usual_high": (1000, "500 mg every 6 hours or 1 g every 12 hours")}),
 dict(drug="vancomycin", route="INTRAVENOUS", scope="day", unit="mg", limits={
    "usual_high": (2000, "usual daily intravenous dose is 2 g divided"),
    "hard_high":  (2000, "total daily dosage should not exceed 2 g")}),
 dict(drug="vancomycin", route="ORAL", scope="day", unit="mg", limits={
    "hard_high":  (2000, "total daily dosage should not exceed 2 g")}),
 dict(drug="lorazepam", route="ORAL", scope="day", unit="mg", limits={
    "usual_low":  (2, "usual range is 2 to 6 mg/day given in divided doses"),
    "usual_high": (6, "usual range is 2 to 6 mg/day given in divided doses"),
    "hard_high":  (10, "daily dosage may vary from 1 to 10 mg/day")}),
 dict(drug="lorazepam", route="INTRAVENOUS", scope="dose", unit="mg", limits={
    "usual_high": (4, "usual recommended dose of lorazepam injection is 4 mg given slowly")}),
 dict(drug="prednisone", route="ORAL", scope="day", unit="mg", limits={
    "usual_low":  (5, "initial dosage of prednisone may vary from 5 to 60 mg per day"),
    "usual_high": (60, "initial dosage of prednisone may vary from 5 to 60 mg per day")}),
 dict(drug="warfarin", route="ORAL", scope="day", unit="mg", limits={
    "usual_high": (10, "typical maintenance doses are 2 to 10 mg once daily")}),
 dict(drug="potassium_chloride", route="ORAL", scope="dose", unit="mEq", limits={
    "usual_high": (20, "no more than 20 meq is given in a single dose"),
    "hard_high":  (40, "no more than 40 meq is given in a single dose")}),
 dict(drug="potassium_chloride", route="ORAL", scope="day", unit="mEq", limits={
    "usual_high": (100, "maximum daily doses should not exceed 100 meq"),
    "hard_high":  (200, "total daily dose should not exceed 200 meq in a 24 hour period")}),
 dict(drug="potassium_chloride", route="INTRAVENOUS", scope="day", unit="mEq", limits={
    "hard_high":  (200, "24 hour total dose should not exceed 200 meq")}),
 dict(drug="magnesium_sulfate", route="INTRAVENOUS", scope="dose", unit="g", limits={
    "usual_low":  (1, "usual adult dose is 1 g administered im or iv"),
    "usual_high": (4, "initial iv dose of 4 g"),
    "hard_high":  (14, "total initial dose is 10 to 14 g of magnesium sulfate")}),
 dict(drug="magnesium_sulfate", route="INTRAVENOUS", scope="day", unit="g", limits={
    "usual_high": (30, "total daily 24 hr dose of 30 to 40 g should not be exceeded"),
    "hard_high":  (40, "total daily 24 hr dose of 30 to 40 g should not be exceeded")}),
 dict(drug="acetaminophen", route="INTRAVENOUS", scope="dose", unit="mg", limits={
    "usual_high": (1000, "1000 mg every 6 hours or 650 mg every 4 hours to a maximum of 4000 mg per day")}),
 dict(drug="acetaminophen", route="INTRAVENOUS", scope="day", unit="mg", limits={
    "hard_high":  (4000, "maximum of 4000 mg per day")}),
]

# drugs with NO numeric dose rule yet (weak/weight/titration/patch dosing): unit + condition checks only
PENDING_DOSE_RULE = ["heparin", "morphine", "fentanyl", "acetaminophen (oral)"]

# ---- verification: how many cached labels (same drug + route group) contain each quote? ----
def load_group(drug, route):
    labels = json.load(open(CACHE_DIR / f"{drug}.json", encoding="utf-8"))
    inc, exc = LABEL_FILTERS.get(drug, ([], []))
    out = []
    for l in labels:
        text = l["sections"].get("dosage_and_administration", "")
        low = text.lower()
        if not text or route not in l["route"]:
            continue
        if (inc and not any(t in low for t in inc)) or any(t in low for t in exc):
            continue
        out.append(norm(text))
    return out

rows, cache_groups = [], {}
for e in DOSE_TABLE:
    key = (e["drug"], e["route"])
    if key not in cache_groups:
        cache_groups[key] = load_group(*key)
    group = cache_groups[key]
    for field, (value, quote) in e["limits"].items():
        nq = norm(quote)
        support = sum(1 for t in group if nq in t)
        pct = support / len(group) if group else 0.0
        status = "OK" if support >= 3 and pct >= 0.10 else ("LOW_SUPPORT" if support > 0 else "NOT_FOUND")
        rows.append(dict(drug=e["drug"], route=e["route"], scope=e["scope"], unit=e["unit"], field=field,
                         value=value, labels_in_group=len(group), labels_with_quote=support,
                         pct=round(pct, 2), status=status, quote=quote))

limits_df = pd.DataFrame(rows)
limits_df.to_csv(PROFILE_DIR / "dose_limits_verification.csv", index=False)
pd.set_option("display.width", 220, "display.max_colwidth", 60, "display.max_rows", 100)
display(limits_df.drop(columns=["quote"]))

print("\nSTATUS COUNTS:", limits_df["status"].value_counts().to_dict())
print("Drugs with no numeric dose rule yet (unit + condition checks only):", PENDING_DOSE_RULE)

# ---- save ONLY verified limits (with their evidence quotes) for the dose checker ----
verified = [r for r in rows if r["status"] == "OK"]
(PROJECT_DIR / "data").mkdir(exist_ok=True)
json.dump({"route_map": ROUTE_MAP, "mass_to_mg": MASS_TO_MG,
           "high_policy_note": "WARNING = outside label usual range; HIGH = above label-stated maximum",
           "limits": verified},
          open(PROJECT_DIR / "data" / "dose_table.json", "w"), indent=1)
print(f"Saved {len(verified)} verified limits to data/dose_table.json")

,drug,route,scope,unit,field,value,labels_in_group,labels_with_quote,pct,status
0,furosemide,ORAL,dose,mg,usual_low,20,97,63,0.65,OK
1,furosemide,ORAL,dose,mg,usual_high,80,97,63,0.65,OK
2,furosemide,ORAL,day,mg,usual_high,80,97,97,1.00,OK
3,furosemide,ORAL,day,mg,hard_high,600,97,97,1.00,OK
4,furosemide,INTRAVENOUS,dose,mg,usual_low,20,47,44,0.94,OK
5,furosemide,INTRAVENOUS,dose,mg,usual_high,40,47,44,0.94,OK
6,metoprolol,ORAL,day,mg,usual_low,100,147,119,0.81,OK
7,metoprolol,ORAL,day,mg,usual_high,400,147,119,0.81,OK
8,metoprolol,ORAL,day,mg,hard_high,450,147,119,0.81,OK
9,metoprolol,INTRAVENOUS,dose,mg,usual_high,5,20,20,1.00,OK



STATUS COUNTS: {'OK': 34}
Drugs with no numeric dose rule yet (unit + condition checks only): ['heparin', 'morphine', 'fentanyl', 'acetaminophen (oral)']
Saved 34 verified limits to data/dose_table.json
